In [3]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
iris = load_iris()
X = iris.data
y = iris.target
print("Dataset Shape:", X.shape)
print("Target Classes:", iris.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)

def calculate_mean_std(X, y):
    mean = {}
    std = {}
    classes = np.unique(y)
    for c in classes:
        X_class = X[y == c]
        mean[c] = np.mean(X_class, axis=0)
        std[c] = np.std(X_class, axis=0)
    return mean, std
mean, std = calculate_mean_std(X_train, y_train)

def calculate_prior(y):
    classes, counts = np.unique(y, return_counts=True)
    prior = {}
    total = len(y)
    for c, count in zip(classes, counts):
        prior[c] = count / total
    return prior
prior = calculate_prior(y_train)

def gaussian_probability(x, mean, std):

    exponent = np.exp(
        -((x - mean) ** 2) / (2 * std ** 2)
    )
    probability = (
        1 / (np.sqrt(2 * np.pi) * std)
    ) * exponent
    return probability

def predict_sample(sample, mean, std, prior):

    probabilities = {}
    classes = mean.keys()
    for c in classes:
    
        probability = prior[c]
        for i in range(len(sample)):

            feature_probability = gaussian_probability(
                sample[i],
                mean[c][i],
                std[c][i]
            )
            probability = probability * feature_probability
        probabilities[c] = probability
    predicted_class = max(
        probabilities,
        key=probabilities.get
    )
    return predicted_class
y_pred = []
for sample in X_test:

    prediction = predict_sample(
        sample,
        mean,
        std,
        prior
    )

    y_pred.append(prediction)

print("\nActual Values:")
print(y_test)
print("\nPredicted Values:")
print(y_pred)

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:")
print(accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Dataset Shape: (150, 4)
Target Classes: ['setosa' 'versicolor' 'virginica']

Training Data: (120, 4)
Testing Data: (30, 4)

Actual Values:
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]

Predicted Values:
[1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2, 0, 2, 2, 2, 2, 2, 0, 0]

Accuracy:
1.0

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
